# 7. Model Export & Customer Scoring

In the last notebook we decided **Random Forest with threshold tuning** is the best model.  
In this notebook we make the model actually usable:

### Steps:
- Train the final Random Forest model (same settings as notebook 04)
- Save the model and scaler as `.pkl` files in `models/`
- Give every customer a churn probability and a risk band
- Export the scored data for **PostgreSQL, Power BI and Excel**

## 7.1 Load Data & Prepare Features

Same preparation as notebooks 03 and 04:
- Gender → 0/1
- Country → one-hot (drop_first)
- Split first, then scale (fit scaler on train only)

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier

df = pd.read_csv("../data/raw/Bank Customer Churn Prediction.csv")
raw_df = df.copy()   # keep original columns for the export later

df['gender'] = df['gender'].map({'Male': 0, 'Female': 1})
df = pd.get_dummies(df, columns=['country'], drop_first=True)

X = df.drop(['churn', 'customer_id'], axis=1)
y = df['churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

num_cols = ['age', 'credit_score', 'balance', 'estimated_salary']
scaler = StandardScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

print(X_train.shape, X_test.shape)
list(X.columns)

(8000, 11) (2000, 11)


['credit_score',
 'gender',
 'age',
 'tenure',
 'balance',
 'products_number',
 'credit_card',
 'active_member',
 'estimated_salary',
 'country_Germany',
 'country_Spain']

## 7.2 Train Final Model & Choose Threshold

From notebook 04 (threshold tuning on the test set):

| Threshold | Churn Precision | Churn Recall |
|---|---|---|
| 0.5 | 0.76 | 0.46 |
| 0.4 | 0.68 | 0.56 |
| **0.3** | **0.57** | **0.67** |

For a bank, **missing a churner is worse than a false alarm** (a retention call is cheap, losing a customer is expensive).  
So I am using **threshold = 0.3**, which catches about 2 out of 3 churners.

In [2]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

THRESHOLD = 0.3

rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X_train, y_train)

y_prob = rf.predict_proba(X_test)[:, 1]
y_pred = (y_prob >= THRESHOLD).astype(int)

print("ROC-AUC:", round(roc_auc_score(y_test, y_prob), 3))
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

ROC-AUC: 0.854
              precision    recall  f1-score   support

           0       0.91      0.87      0.89      1593
           1       0.57      0.67      0.62       407

    accuracy                           0.83      2000
   macro avg       0.74      0.77      0.75      2000
weighted avg       0.84      0.83      0.83      2000

[[1383  210]
 [ 133  274]]


### ✔ Observations
- ROC-AUC stays the same as notebook 04 (same model, same split).
- With threshold 0.3 the model catches more churners, but also gives more false alarms.
- This is a business trade-off, not only a technical one (more on this in `docs/PRODUCT_BRIEF.md`).

## 7.3 Save Model & Scaler (.pkl)

Saving with `pickle` so that `scripts/predict.py` can load the model without training again.

In [3]:
import pickle
import os

os.makedirs("../models", exist_ok=True)

with open("../models/churn_model.pkl", "wb") as f:
    pickle.dump(rf, f)

with open("../models/scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

print("Saved:", os.listdir("../models"))

Saved: ['scaler.pkl', 'churn_model.pkl']


In [4]:
# quick check - load the saved files back and predict one customer
with open("../models/churn_model.pkl", "rb") as f:
    loaded_model = pickle.load(f)

print("Prediction from loaded model:", loaded_model.predict_proba(X_test.iloc[[0]])[:, 1])
print("Prediction from original model:", rf.predict_proba(X_test.iloc[[0]])[:, 1])

Prediction from loaded model: [0.03]


Prediction from original model: [0.03]


## 7.4 Score Every Customer (for Dashboard)

Problem: if I score the training customers with a model that was trained on them, the scores will look too perfect.

Solution: use `cross_val_predict` (5 folds). Each customer gets a score from a model that **did not see that customer** during training.  
(Random Forest doesn't need scaling, so I use the unscaled `X` here.)

In [5]:
from sklearn.model_selection import cross_val_predict

rf_cv = RandomForestClassifier(n_estimators=200, random_state=42)
churn_prob = cross_val_predict(rf_cv, X, y, cv=5, method='predict_proba')[:, 1]

scored = raw_df.copy()
scored['churn_probability'] = churn_prob.round(4)
scored['predicted_churn'] = (scored['churn_probability'] >= THRESHOLD).astype(int)

def risk_band(p):
    if p >= 0.6:
        return 'High'
    elif p >= THRESHOLD:
        return 'Medium'
    else:
        return 'Low'

scored['risk_band'] = scored['churn_probability'].apply(risk_band)
scored['risk_band'].value_counts()

risk_band
Low       7655
Medium    1390
High       955
Name: count, dtype: int64

## 7.5 Add Simple Groups for Reporting

Dashboards are easier to read with groups instead of raw numbers:
- Age group
- Balance group
- Credit score group

In [6]:
scored['age_group'] = pd.cut(scored['age'], bins=[17, 29, 39, 49, 59, 100],
                            labels=['18-29', '30-39', '40-49', '50-59', '60+'])

scored['balance_group'] = pd.cut(scored['balance'], bins=[-1, 0, 50000, 100000, 150000, 300000],
                                labels=['0', '1-50K', '50K-100K', '100K-150K', '150K+'])

scored['credit_score_group'] = pd.cut(scored['credit_score'], bins=[0, 579, 669, 739, 799, 900],
                                     labels=['Poor (<580)', 'Fair (580-669)', 'Good (670-739)',
                                             'Very Good (740-799)', 'Excellent (800+)'])
scored.head()

,customer_id,credit_score,country,gender,age,tenure,balance,products_number,credit_card,active_member,estimated_salary,churn,churn_probability,predicted_churn,risk_band,age_group,balance_group,credit_score_group
0,15634602,619,France,Female,42,2,0.00,1,1,1,101348.88,1,0.425,1,Medium,40-49,0,Fair (580-669)
1,15647311,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0,0.255,0,Low,40-49,50K-100K,Fair (580-669)
2,15619304,502,France,Female,42,8,159660.80,3,1,0,113931.57,1,0.870,1,High,40-49,150K+,Poor (<580)
3,15701354,699,France,Female,39,1,0.00,2,0,0,93826.63,0,0.065,0,Low,30-39,0,Good (670-739)
4,15737888,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0,0.140,0,Low,40-49,100K-150K,Excellent (800+)


In [7]:
# Check: does the risk band actually match real churn?
scored.groupby('risk_band')['churn'].agg(['count', 'mean']).rename(columns={'mean': 'actual_churn_rate'})

,count,actual_churn_rate
risk_band,,
High,955,0.837696
Low,7655,0.088700
Medium,1390,0.401439


## 7.6 Export Scored Data

This one CSV is used by:
- `sql/` → loaded into PostgreSQL
- `powerbi/` → Power BI dashboard
- `excel/` → summary workbook

In [8]:
os.makedirs("../data/processed", exist_ok=True)
scored.to_csv("../data/processed/customers_scored.csv", index=False)
print("Exported:", scored.shape)

Exported: (10000, 18)


## ✔ Summary

- Final model: **Random Forest (200 trees), threshold 0.3**
- Saved `models/churn_model.pkl` and `models/scaler.pkl`
- Every customer now has a **churn probability** and a **risk band** (High / Medium / Low)
- `data/processed/customers_scored.csv` is ready for SQL, Power BI and Excel

**Next:** use `scripts/predict.py` to score new customers.